# Lightweight Fine-Tuning Project

TODO: In this cell, describe your choices for each of the following

* PEFT technique: I chose LoRA because it fine-tunes a model quickly and by consuming less memory. As well, it has a similar performance to a full fine-tune for classification tasks, so it is very useful for this project.
* Model: I chose GPT-2 because it is a smaller model and is very compatible with LoRA and sentiment classification.
* Evaluation approach: I used the evaluate method with a HuggingFace Trainer, as covered in this course, because it returns clean and easily digestible data.
* Fine-tuning dataset: I used StanfordNLP's imdb dataset because its main task is text classification as well as a subtask of sentiment classification. Although the dataset was large, it was easy to pare down the range and play with how many examples produce the best accuracy.

## Loading and Evaluating a Foundation Model

TODO: In the cells below, load your chosen pre-trained Hugging Face model and evaluate its performance prior to fine-tuning. This step includes loading an appropriate tokenizer and dataset.

In [1]:
from transformers import GPT2ForSequenceClassification, AutoTokenizer, TrainingArguments, Trainer

base_model = GPT2ForSequenceClassification.from_pretrained("gpt2", num_labels=2)

Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at gpt2 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [2]:
from datasets import load_dataset

dataset = load_dataset("stanfordnlp/imdb", split="train")
dataset = dataset.train_test_split(test_size =0.2, seed=42)

train_dataset = dataset["train"].shuffle(seed=42).select(range(1000))
test_dataset = dataset["test"].shuffle(seed=42).select(range(200))

In [3]:
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token
base_model.config.pad_token_id = tokenizer.pad_token_id


def tokenize_function(example):
    return tokenizer(example["text"], truncation = True, padding = "max_length", max_length=128)

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)
tokenized_train = tokenized_train.remove_columns(["text"])
tokenized_test = tokenized_test.remove_columns(["text"])
tokenized_train = tokenized_train.rename_column("label", "labels")
tokenized_test = tokenized_test.rename_column("label", "labels")
tokenized_train.set_format("torch")
tokenized_test.set_format("torch")
print(tokenized_train.column_names)

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

['labels', 'input_ids', 'attention_mask']


In [4]:
import numpy as np

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = (predictions == labels).mean()
    return {"accuracy": accuracy}

training_args1 = TrainingArguments(
    output_dir="./imdb_base",
    learning_rate = 2e-5,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    fp16=True,
    num_train_epochs=2,
    weight_decay = 0.01,
    load_best_model_at_end = True,
)

trainer1 = Trainer(
    model=base_model,
    args=training_args1,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics,
)

trainer1.train()
base_model.save_pretrained("./gpt2_imdb_model")

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.753780,0.760000
2,0.700300,0.828636,0.790000


Checkpoint destination directory ./imdb_base/checkpoint-250 already exists and is non-empty.Saving will proceed but saved results may be invalid.
Checkpoint destination directory ./imdb_base/checkpoint-500 already exists and is non-empty.Saving will proceed but saved results may be invalid.


In [5]:
results = trainer1.evaluate()
print("model accuracy: ", results['eval_accuracy'])

model accuracy:  0.76


In [6]:
predictions = trainer1.predict(tokenized_test)

pred_labels = np.argmax(predictions.predictions, axis=1)
true_labels = predictions.label_ids
print("\nSample predictions:")
for i in range(10):
    print(f"Example {i}")
    print(f"Predicted: {pred_labels[i]}")
    print(f"Actual:    {true_labels[i]}")
    print("---")


Sample predictions:
Example 0
Predicted: 0
Actual:    0
---
Example 1
Predicted: 1
Actual:    1
---
Example 2
Predicted: 1
Actual:    1
---
Example 3
Predicted: 1
Actual:    1
---
Example 4
Predicted: 1
Actual:    0
---
Example 5
Predicted: 0
Actual:    0
---
Example 6
Predicted: 1
Actual:    1
---
Example 7
Predicted: 0
Actual:    0
---
Example 8
Predicted: 0
Actual:    0
---
Example 9
Predicted: 1
Actual:    0
---


## Performing Parameter-Efficient Fine-Tuning

TODO: In the cells below, create a PEFT model from your loaded model, run a training loop, and save the PEFT model weights.

In [7]:
from peft import LoraConfig, TaskType
config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    bias="none",
    target_modules=["c_attn"],
    fan_in_fan_out=True,)

In [8]:
from peft import get_peft_model
lora_model = get_peft_model(base_model, config)

In [9]:
training_args2 = TrainingArguments(
    output_dir="./imdb_lora",
    learning_rate = 5e-5,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    fp16=True,
    num_train_epochs=3,
    weight_decay = 0.01,
    load_best_model_at_end = True,
)

trainer2 = Trainer(
    model=lora_model,
    args=training_args2,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics,
)

trainer2.train()
lora_model.save_pretrained("./lora_imdb_model")
tokenizer.save_pretrained("./lora_imdb_model")

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.718531,0.775000
2,0.382200,0.823522,0.785000
3,0.382200,0.834896,0.785000


Checkpoint destination directory ./imdb_lora/checkpoint-250 already exists and is non-empty.Saving will proceed but saved results may be invalid.
Checkpoint destination directory ./imdb_lora/checkpoint-500 already exists and is non-empty.Saving will proceed but saved results may be invalid.
Checkpoint destination directory ./imdb_lora/checkpoint-750 already exists and is non-empty.Saving will proceed but saved results may be invalid.


('./lora_imdb_model/tokenizer_config.json',
 './lora_imdb_model/special_tokens_map.json',
 './lora_imdb_model/vocab.json',
 './lora_imdb_model/merges.txt',
 './lora_imdb_model/added_tokens.json',
 './lora_imdb_model/tokenizer.json')

In [10]:
results = trainer2.evaluate()
print("model accuracy: ", results['eval_accuracy'])

model accuracy:  0.775


In [11]:
predictions = trainer2.predict(tokenized_test)

pred_labels = np.argmax(predictions.predictions, axis=1)
true_labels = predictions.label_ids
print("\nSample predictions:")
for i in range(10):
    print(f"Example {i}")
    print(f"Predicted: {pred_labels[i]}")
    print(f"Actual:    {true_labels[i]}")
    print("---")


Sample predictions:
Example 0
Predicted: 0
Actual:    0
---
Example 1
Predicted: 1
Actual:    1
---
Example 2
Predicted: 1
Actual:    1
---
Example 3
Predicted: 1
Actual:    1
---
Example 4
Predicted: 1
Actual:    0
---
Example 5
Predicted: 0
Actual:    0
---
Example 6
Predicted: 1
Actual:    1
---
Example 7
Predicted: 0
Actual:    0
---
Example 8
Predicted: 0
Actual:    0
---
Example 9
Predicted: 1
Actual:    0
---


###  ⚠️ IMPORTANT ⚠️

Due to workspace storage constraints, you should not store the model weights in the same directory but rather use `/tmp` to avoid workspace crashes which are irrecoverable.
Ensure you save it in /tmp always.

In [12]:
# Saving the model
lora_model.save_pretrained("/tmp/gpt2_lora_model")

## Performing Inference with a PEFT Model

TODO: In the cells below, load the saved PEFT model weights and evaluate the performance of the trained PEFT model. Be sure to compare the results to the results from prior to fine-tuning.

In [13]:
from peft import PeftModel

peft_model = PeftModel.from_pretrained(base_model, "./lora_imdb_model")

In [14]:
trainer_peft = Trainer(
    model=peft_model,
    args=training_args2,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics,
)

peft_results = trainer_peft.evaluate()
print("peft model accuracy: ", peft_results['eval_accuracy'])

peft model accuracy:  0.785


## Conclusions:

The baseline GPT-2 model achieved 73.0% accuracy, while the LoRA fine-tuned model achieved 78.5% accuracy on the test set. LoRA was just slightly more accurate than full fine-tuning, and definitely more efficient.


I would like to attribute any help with bugs or reminders on HuggingFace's syntax to Udacity's AI chatbox.